# Week 5: Sequence to function

| | |
|---|---|
| **Due** | start of the Week 6 meeting |
| **Prerequisites** | Weeks 1–4 (you can read a BLAST report and build an MSA) |
| **Produces** | Your annotation-transfer record and propagation account, in this notebook |
| **Also due this week** | **Part I checkpoint memo — one page.** |

## Learning objectives

1. Distinguish an EC (Enzyme Commission) number from a GO (Gene Ontology) term, and state what each can and cannot express.
2. Explain why GO is a directed acyclic graph (DAG) rather than a tree, and apply the true-path rule to a term with two parents.
3. Classify a GO annotation by its evidence code and say what IEA (Inferred from Electronic Annotation) does not guarantee.
4. Transfer a function by homology and qualify the transfer with identity, coverage, and the evidence behind the source annotation.
5. Compare an alignment-based notion of relatedness against a protein language model's, and argue
   which to believe where they disagree.
6. Trace a misannotation propagation chain and name the mechanism that let it spread *(Biology
   extension)*.

## Where this sits

For three weeks you have been rewarded for finding similar sequences: align two, score the substitutions, search fast, build a profile. Every one of those exercises trained a move you have not yet been asked to justify — taking the label off a hit and putting it on your query.

This week names that move and then breaks it. Similarity is evidence about **ancestry**. Function is a claim about **chemistry and context**. The step between them is an inference, and it is an inference that the reference databases have made hundreds of millions of times without anyone measuring anything.

You are not being taught to distrust annotation. You are being taught to read an annotation and know what kind of claim it is. That reflex becomes load-bearing in Week 9, when you annotate a whole panel of proteins, and again in Week 13, when you have to decide what counts as ground truth for your own screen.

In [1]:
#@title Setup — run this first
%pip install -q "biopython>=1.85" "pandas>=2.0" "matplotlib>=3.7" "requests>=2.31"
print("setup complete")

Note: you may need to restart the kernel to use updated packages.
setup complete


In [2]:
import pathlib, requests
FIXTURES = pathlib.Path("fixtures")
COURSE_DATA_URL = None   

def course_file(name: str) -> pathlib.Path:
    local = FIXTURES / name
    if local.exists():
        return local
    if COURSE_DATA_URL:
        FIXTURES.mkdir(exist_ok=True)
        r = requests.get(f"{COURSE_DATA_URL}/{name}", timeout=30); r.raise_for_status()
        local.write_bytes(r.content); return local
    raise FileNotFoundError(f"{name} not found. Put fixtures/ beside this notebook.")

print("fixtures:", FIXTURES.resolve())

fixtures: C:\Users\laura\OneDrive - James Madison University\Documents\ISAT\454\compbio\fixtures


---
## Part 1: What a functional annotation claims

"Function" is at least three different questions, and the vocabularies answer different ones.

**EC numbers** describe *a reaction*. Four digits, each narrowing the last. They belong to the chemistry, not to the protein: two proteins with no common ancestor that catalyse the same reaction get the same EC number, and a protein that catalyses two reactions gets two.

**GO terms** describe *a protein*, in three separate aspects: molecular function (what it does at the bench), biological process (what it is for), cellular component (where it is). GO terms have identifiers of the form `GO:0004146` and are arranged in a graph, which is the subject of the next section.

Run the cell below and read `1.5.1.3` digit by digit.

In [3]:
import json
from pathlib import Path

cache = json.loads(course_file("verified_cache.json").read_text())
rec = cache["P0ABQ4"]

print(f'{rec["entry_name"]}  {rec["protein_name"]}  ({rec["length"]} aa)')
print(f'EC {rec["ec_number"]}\n')

EC_LEVELS = [
    ("1",       "class",       "oxidoreductase - transfers electrons"),
    ("1.5",     "subclass",    "acting on the CH-NH group of donors"),
    ("1.5.1",   "sub-subclass","with NAD+ or NADP+ as acceptor"),
    ("1.5.1.3", "serial",      "dihydrofolate reductase - this exact reaction"),
]
for code, level, meaning in EC_LEVELS:
    print(f"  {code:<9} {level:<13} {meaning}")

print("""
Notice the asymmetry. Digit 1 is a statement about the broad chemistry and is
hard to get wrong. Digit 4 indicates the precise protein from a sub-subclass.""")

DYR_ECOLI  Dihydrofolate reductase  (159 aa)
EC 1.5.1.3

  1         class         oxidoreductase - transfers electrons
  1.5       subclass      acting on the CH-NH group of donors
  1.5.1     sub-subclass  with NAD+ or NADP+ as acceptor
  1.5.1.3   serial        dihydrofolate reductase - this exact reaction

Notice the asymmetry. Digit 1 is a statement about the broad chemistry and is
hard to get wrong. Digit 4 names a substrate, and it is the digit that a
homology transfer is least entitled to.


### Why GO is a graph and not a tree

A tree gives every node one parent. GO does not, because biology does not: a term can be a *kind of* one thing and a *part of* another at the same time. So GO is a **directed acyclic graph** (i.e., DAG) — arrows point from specific terms toward general ones, a term may have several parents, and no path ever loops back on itself.

The consequence you must be able to apply is the **true-path rule**: annotating a protein to a term annotates it to *every ancestor of that term*, along every path. If you say a protein does X, you have also said it does everything X is a kind of, and is part of everything X is part of.

Why do we care? First, it means a specific annotation is a much larger claim than it looks. Second, it is why you cannot evaluate function prediction with the flat multi-label accuracy you may have seen elsewhere: a predictor that only says "binds something" gets a lot of ancestor terms right for free.

The graph below is a small hand-built fragment written for this exercise, not a download of the real ontology. It is here so you can see the structure; the term identifiers are real but the fragment is deliberately tiny.

In [4]:
# A hand-built fragment of GO structure. Edges point child -> parent.
# Relation types are kept because they are not interchangeable.
GO_EDGES = {
    "GO:0004146": [("GO:0016646", "is_a")],                       # dihydrofolate reductase activity
    "GO:0016646": [("GO:0016645", "is_a")],                       # oxidoreductase, CH-NH, NAD(P) acceptor
    "GO:0016645": [("GO:0016491", "is_a")],                       # oxidoreductase, CH-NH group of donors
    "GO:0016491": [("GO:0003824", "is_a")],                       # oxidoreductase activity
    "GO:0003824": [("GO:0003674", "is_a")],                       # catalytic activity -> molecular_function
    "GO:0003674": [],
    # A term with TWO parents, by two different relations:
    "GO:0046654": [("GO:0009396", "part_of"), ("GO:0006760", "is_a")],   # tetrahydrofolate biosynthetic process
    "GO:0009396": [("GO:0008152", "is_a")],                       # folic acid-containing compound biosynthesis
    "GO:0006760": [("GO:0008152", "is_a")],                       # folic acid-containing compound metabolism
    "GO:0008152": [("GO:0008150", "is_a")],                       # metabolic process -> biological_process
    "GO:0008150": [],
}

NAMES = {
    "GO:0004146": "dihydrofolate reductase activity",
    "GO:0016646": "oxidoreductase activity, CH-NH, NAD(P) as acceptor",
    "GO:0016645": "oxidoreductase activity, acting on the CH-NH group of donors",
    "GO:0016491": "oxidoreductase activity",
    "GO:0003824": "catalytic activity",
    "GO:0003674": "molecular_function (root)",
    "GO:0046654": "tetrahydrofolate biosynthetic process",
    "GO:0009396": "folic acid-containing compound biosynthetic process",
    "GO:0006760": "folic acid-containing compound metabolic process",
    "GO:0008152": "metabolic process",
    "GO:0008150": "biological_process (root)",
}

def parents(term):
    return GO_EDGES.get(term, [])

print("GO:0046654 has", len(parents("GO:0046654")), "parents:")
for p, rel in parents("GO:0046654"):
    print(f"   --{rel:<8}-> {p}  {NAMES[p]}")


GO:0046654 has 2 parents:
   --part_of -> GO:0009396  folic acid-containing compound biosynthetic process
   --is_a    -> GO:0006760  folic acid-containing compound metabolic process

A tree would force you to pick one. The DAG does not, and neither does the biology.


In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a function ancestors(term) returning every ancestor of a GO term along every path, plus the full annotated set for GO:0046654
# Example: ancestors('GO:0004146') should return 5 terms ending at the molecular_function root; ancestors('GO:0046654') must include BOTH GO:0009396 and GO:0006760
# ---- END ----


---
## Part 2: Evidence codes

Every GO annotation carries a three-letter **evidence code** saying what the source of evidence is for that annotation. The code is not metadata about the annotation; on most days it is more informative than the annotation.

| Class | Examples | Who was in the loop |
|---|---|---|
| Experimental | IDA (Inferred from Direct Assay), IMP (Inferred from Mutant Phenotype), IPI (Inferred from Physical Interaction) | Someone did an experiment; a curator read the paper |
| Phylogenetic / sequence, curated | ISS (Inferred from Sequence or Structural Similarity), IBA (Inferred from Biological Aspect of Ancestor) | A curator made a similarity judgement and signed it |
| Author statement | TAS, NAS | A paper asserted it; TAS cites a source, NAS does not |
| **Automatic** | **IEA (Inferred from Electronic Annotation)** | **Nobody. A pipeline transferred it.** |

IEA is the only class with no curator anywhere in it, and IEA annotations are the overwhelming majority of GO annotations in the databases — by a wide margin, though the exact share moves with every release, so quote it with a release date or not at all. `[VERIFY the current share before quoting a number.]`

An IEA annotation is not wrong by default. It is *unreviewed*, which is a different property, and it is the property that the rest of this notebook is about. The step from "a pipeline decided this" to "this is known" happens silently, in the reading.

In [ ]:
# --- OPTIONAL NETWORK CELL -------------------------------------------------
# Fetches the live GO annotations for P0ABQ4 with their evidence codes.
# Nothing later in this notebook depends on it. If it fails, keep going.
import pandas as pd

go_table = None
try:
    import requests
    url = "https://rest.uniprot.org/uniprotkb/P0ABQ4.json"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    rows = []
    for xref in data.get("uniProtKBCrossReferences", []):
        if xref.get("database") != "GO":
            continue
        props = {p["key"]: p["value"] for p in xref.get("properties", [])}
        term = props.get("GoTerm", "")
        evidence = props.get("GoEvidenceType", "")
        rows.append({
            "go_id": xref.get("id"),
            "aspect": term[:1],                       # C, F or P
            "term": term[2:],
            "evidence": evidence.split(":")[0],
        })
    go_table = pd.DataFrame(rows).sort_values(["evidence", "aspect"])
    print(go_table.to_string(index=False))
    print("\ncounts by evidence code:")
    print(go_table["evidence"].value_counts().to_string())
except Exception as e:
    print("live fetch unavailable:", type(e).__name__, e)
    print("Continue - the rest of the notebook is offline. If you can run this later,")
    print("do, and report the ratio of reviewed to IEA codes you saw, with the date.")

---
## Part 3 (Core): Transfer an annotation

Here is the entire method:

```
1. search your query against a database
2. take the best hit
3. copy its function onto your query
```

That is homology transfer. It is also, by a distance, the strongest single baseline in the CAFA
(Critical Assessment of Functional Annotation) competitions — many far more elaborate methods have
failed to beat it. Both things are true: it works well, and it is how the databases became partly wrong.

The difference between a defensible transfer and an indefensible one is not the method. It is
whether you wrote down the qualifications. A transfer record must state:

- **identity** over the aligned region, and **coverage** — what fraction of your query was aligned at all;
- **which residues** matched, specifically whether the catalytic ones did;
- **how specific** the transferred term is (EC digit 4 versus digit 1; a leaf GO term versus a mid-graph one);
- **the evidence code on the source annotation** — if the hit's own annotation is IEA, you are transferring from a transfer;
- **what you would have to see to withdraw the claim.**

The cell below is fully offline. It computes global pairwise identity between *E. coli* DHFR
(`P0ABQ4`, the annotated reference) and each of the twelve reviewed bacterial DHFRs in the fixture,
plus human DHFR (`P00374`) as an outgroup.

In [ ]:
# Offline. Depends only on the fixtures folder
from Bio import SeqIO, Align
from Bio.Align import substitution_matrices
import pandas as pd

aligner = Align.PairwiseAligner(mode="global", open_gap_score=-11, extend_gap_score=-1)
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")

query = next(SeqIO.parse(course_file("dhfr_ecoli.fasta"), "fasta"))

def identity_and_coverage(a_seq, b_seq):
    """Percent identity over aligned columns, and coverage of a_seq."""
    aln = aligner.align(a_seq, b_seq)[0]
    top, bottom = aln[0], aln[1]
    ident  = sum(1 for x, y in zip(top, bottom) if x == y and x != "-")
    shared = sum(1 for x, y in zip(top, bottom) if x != "-" and y != "-")
    return 100 * ident / shared, 100 * shared / len(a_seq), aln.score

rows = []
subjects = list(SeqIO.parse(course_file("dhfr_homologs.fasta"), "fasta"))
subjects.append(next(SeqIO.parse(course_file("dhfr_human.fasta"), "fasta")))

for rec in subjects:
    acc, name = rec.id.split("|")[1], rec.id.split("|")[2]
    pid, cov, score = identity_and_coverage(query.seq, rec.seq)
    rows.append({"accession": acc, "entry": name, "length": len(rec.seq),
                 "pct_identity": round(pid, 1), "coverage_pct": round(cov, 1),
                 "blosum62_score": round(score, 1)})

hits = pd.DataFrame(rows).sort_values("pct_identity", ascending=False).reset_index(drop=True)
print(hits.to_string(index=False))

print("""
Two things to look at before you go on.

(1) Three rows sit at 100% identity: P0ABQ4, P0ABQ5, P0ABQ6. They are the same sequence under
    three accessions. If you counted hits as independent evidence, you just counted one sequence
    three times - the same redundancy trap as Week 4, now with consequences for a function call.

(2) Human DYR_HUMAN is around 32% identity to the E. coli enzyme, squarely inside Rost's twilight
    zone - and it is unambiguously the same EC number, 1.5.1.3. Identity alone would have told you
    to refuse a transfer that happens to be correct.""")

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a documented transfer record for one uncharacterized query: a dict or one-row DataFrame with identity, coverage, catalytic-residue check, term specificity, the source annotation's evidence code, your call, and your stated withdrawal condition
# Example: pick DYR_THEMA (Q60034, ~31% identity to P0ABQ4) as your query and P0ABQ4 as the annotated source; your record should end in a call at the EC 1.5.1 level with an explicit reason for not asserting the fourth digit
# ---- END ----


---
## Part 4 (Core): Your call versus a protein language model

Everything you have done so far this semester rests on alignment. Think back to what you've done already this semester:
You aligned by hand in Week 2, searched with a heuristic in Week 3, built a profile in Week 4, 
and transferred an annotation across
one in Part 3 of Week 5. Alignment has been the instrument the whole way.

A **protein language model** does none of that. ESM-2 was trained on tens of millions of raw
sequences with no alignments, no structures and no functional labels — the training task is simply
to predict masked residues. What it learned about protein families is compressed into its weights
rather than computed at query time from a database.

So this is the last and sharpest of the course's paired comparisons: an explicit, inspectable
alignment against a learned representation you cannot read. Where the two disagree about which
proteins are related, one of them is wrong, and you have to decide which — without being able to
open the second one up.

Kabir et al. (2024) ran this comparison at scale and found that
language models *also* degrade in the twilight zone, just less steeply than alignment does. You are
reproducing a small version of a published result.

The model is **ESM-2** (`esm2_t12_35M_UR50D`, the small checkpoint), run locally via the `fair-esm`
package. Local inference only: the ESM Atlas folding API is not used anywhere in this course.

What this gives you is an embedding — a vector per sequence — not a GO term. So the comparison is
made this way: embed all the sequences, measure cosine similarity in embedding space, and ask whether
the model's notion of "similar" ranks the family the same way percent identity does. Where the
ordering differs, you have a case where an alignment-free method and an alignment disagree about
relatedness, and therefore potentially about function.

**This cell downloads a model (roughly 30 MB) and will not run without network access.** It is
deliberately isolated: nothing before or after it depends on the variables it defines.

In [ ]:
# --- OPTIONAL NETWORK + DOWNLOAD CELL -------------------------------------
# Downloads esm2_t12_35M_UR50D (~30 MB) on first run. Isolated by design:
# no other cell in this notebook uses anything defined here.
esm_ok = False
try:
    import subprocess, sys
    # CPU-only torch. The default wheel pulls a CUDA stack you do not need to embed
    # twelve short peptides, and on a laptop that is a large download for nothing.
    _torch_args = ["fair-esm", "torch"]
    if "google.colab" not in sys.modules:
        # Local machine: ask PyPI for the CPU build explicitly.
        _torch_args = ["fair-esm", "torch", "--index-url",
                       "https://download.pytorch.org/whl/cpu"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_torch_args],
                   check=True)
    import torch, esm
    from Bio import SeqIO
    import numpy as np, pandas as pd

    model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
    model.eval()
    batch_converter = alphabet.get_batch_converter()

    recs = list(SeqIO.parse(course_file("dhfr_homologs.fasta"), "fasta"))
    recs.append(next(SeqIO.parse(course_file("dhfr_human.fasta"), "fasta")))
    data = [(r.id.split("|")[1], str(r.seq)) for r in recs]

    labels, strs, tokens = batch_converter(data)
    with torch.no_grad():
        out = model(tokens, repr_layers=[12], return_contacts=False)
    token_reps = out["representations"][12]

    # Mean-pool over real residues only (skip BOS/EOS and padding).
    embeddings = {}
    for i, (acc, seq) in enumerate(data):
        embeddings[acc] = token_reps[i, 1:len(seq) + 1].mean(0).numpy()

    def cosine(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    ref = embeddings["P0ABQ4"]
    esm_tbl = pd.DataFrame(
        [{"accession": a, "esm_cosine": round(cosine(ref, v), 4)}
         for a, v in embeddings.items()]
    ).sort_values("esm_cosine", ascending=False)
    print(esm_tbl.to_string(index=False))
    esm_ok = True
except Exception as e:
    print("ESM-2 unavailable in this environment:", type(e).__name__, e)
    print("This cell needs network access and a model download. Run it on Colab.")
    print("Nothing else in the notebook depends on it.")

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: REQUIRED (Core): a side-by-side comparison of the ESM-2 cosine ranking and the percent-identity ranking, plus 150+ words on which you believe where they disagree and why
# Example: merge on accession, report Spearman rank correlation between the two orderings, and name the sequence whose rank moves most
# ---- END ----


---
## Extensions

Do at least one. Remember: you need four of each type across the term.

### Computing extension: where does transfer stop being trustworthy?

Using the identity table from Part 3, measure how the reliability of a transferred annotation decays
with percent identity, and then argue for a cutoff.

You know the ground truth here, which is what makes the exercise possible: every sequence in
`dhfr_homologs.fasta` is a reviewed DHFR, and so is human `P00374`. So you can score a transfer at
each level of the EC number and see which digits survive how far down the identity scale.

Concretely: for each hit, ask whether transferring EC 1, EC 1.5, EC 1.5.1 and EC 1.5.1.3 from
`P0ABQ4` would have been correct, bin by identity, and plot agreement against identity per EC level.
Then answer in prose: **at what identity would you stop trusting a transferred annotation, and does
your answer depend on which digit you are transferring?**

There is no number you are supposed to arrive at. Rost's twilight zone — roughly 20–30% identity, the
region where sequences may share a fold with no detectable sequence signal — is the relevant
background, not an answer. Your fixture set is a family of homologs at 31–100% identity, which is a
best case and you should say so: a real decay curve needs non-homologous pairs in it too, and you do
not have any. State that limitation explicitly. A cutoff argued from twelve related sequences is a
cutoff for twelve related sequences.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a per-EC-level agreement analysis binned by percent identity, a plot, and 150+ words arguing a cutoff with its limitations stated
# Example: your plot should show EC level 1 agreeing everywhere and level 4 being the one that is actually at risk; the accompanying text must say why this fixture set overstates reliability
# ---- END ----


---
### Biology extension: how one wrong call becomes a consensus

Schnoes et al. (2009) hand-curated four enzyme superfamilies and asked, of every annotation in
several large databases, whether it was right. Two failure modes came out of it, and they are not
the same thing:

- **Misannotation** — the stated function is wrong.
- **Over-annotation** — the stated function is *more specific than the evidence supports*. The
  protein really is an oxidoreductase; it was labelled with a substrate nobody tested.

Over-annotation is the one that scales, because it is produced by exactly the procedure you just
carried out, minus the qualifications.

Here is the mechanism, and it is worth drawing rather than reading. Sequence A is annotated from an
experiment. Sequence B is deposited, matches A, and inherits A's label — including the fourth EC
digit, which A's experiment did support but B's alignment does not. Sequence C is deposited, matches
B, and inherits from B. C's record does not say "via B, from A". It says the function.

**Annotation is transitive. Evidence is not.** After three hops you have three database entries
that agree, and nothing in any of them marks the agreement as a single source seen three times.
A fourth curator, checking their call against the database, finds independent confirmation that
does not exist.

Simulate it.

In [ ]:
# Offline toy. No claim to be a model of any real database - it is the mechanism, isolated.
import random
random.seed(5)

class Entry:
    def __init__(self, acc, label, specificity, evidence, source=None):
        self.acc, self.label = acc, label
        self.specificity = specificity      # 1 = broad class ... 4 = exact substrate
        self.evidence = evidence            # "EXP" or "IEA"
        self.source = source                # accession this was copied from

db = [Entry("A0", "EC 1.5.1.3", 4, "EXP")]  # one real measurement

def transfer(new_acc, db, over_annotate_prob=0.8):
    """Copy the label from a randomly chosen existing entry. No new evidence is created."""
    parent = random.choice(db)
    keep_specificity = parent.specificity if random.random() < over_annotate_prob \
                       else max(1, parent.specificity - 1)
    return Entry(new_acc, parent.label, keep_specificity, "IEA", source=parent.acc)

for i in range(1, 12):
    db.append(transfer(f"A{i}", db))

print(f"{'entry':<6}{'label':<12}{'spec':<6}{'evidence':<10}{'copied from':<12}root")
for e in db:
    root, hops, cur = e.acc, 0, e
    while cur.source is not None:
        cur = next(x for x in db if x.acc == cur.source)
        hops += 1
    print(f"{e.acc:<6}{e.label:<12}{e.specificity:<6}{e.evidence:<10}"
          f"{str(e.source):<12}{cur.acc} ({hops} hops)")

exp = sum(1 for e in db if e.evidence == "EXP")
print(f"""
{len(db)} entries assert EC 1.5.1.3 at substrate-level specificity.
{exp} of them rests on an experiment. The other {len(db)-exp} rest on that same experiment,
none of them says so, and a search returns all {len(db)} as separate lines.""")

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a written account (150-300 words) of one documented case of misannotation propagation, plus the two structural features of the databases that let it spread
# Example: use a case named in Schnoes et al. (2009) or another you can cite; your account must identify the original over-specific call, the mechanism of spread, and how it was eventually caught
# ---- END ----


---
## Submission checklist

- [ ] `ancestors()` implemented; the two-parent assertion on `GO:0046654` passes
- [ ] You can state, in one sentence, what the true-path rule means for a specific annotation
- [ ] Evidence codes read for `P0ABQ4` (note whether from the live fetch or not; if live, record the date)
- [ ] Pairwise identity table computed against `P0ABQ4` (offline)
- [ ] One documented annotation-transfer record, including coverage, catalytic-residue check, source evidence code, and a stated withdrawal condition
- [ ] **ESM-2 embedding ranking compared against the percent-identity ranking, with 150+ words on which you believe where they disagree** 
- [ ] At least one extension, clearly labelled
- [ ] Notebook runs top to bottom in a fresh runtime, with the network cells failing gracefully
- [ ] **Part I checkpoint memo, one page** (separate submission)
